# Evidence rows

Every number in a review packet must trace back to a row with a stable ID. This
notebook defines that row and how to find it again.

In [ ]:
#| default_exp evidence

In [ ]:
#| export
from collections.abc import Sequence
from dataclasses import dataclass

One computed metric for one series, model, origin and horizon.

In [ ]:
#| export
@dataclass(frozen=True)
class EvidenceRow:
    row_id: str
    series_id: str
    model_version: str
    origin_day: int
    horizon: int
    metric: str
    value: float
    unit: str

Build a row. The ID is made from its keys, so the same inputs always give the same ID.

In [ ]:
#| export
def metric_evidence(
    *,
    series_id: str,
    model_version: str,
    origin_day: int,
    horizon: int,
    metric: str,
    value: float,
    unit: str = "units",
) -> EvidenceRow:
    """Build an evidence row whose ID is stable across re-runs."""
    return EvidenceRow(
        row_id=f"{series_id}__{model_version}__origin_{origin_day}__h{horizon}__{metric}",
        series_id=series_id,
        model_version=model_version,
        origin_day=origin_day,
        horizon=horizon,
        metric=metric,
        value=value,
        unit=unit,
    )

Find the one row matching all keys. Zero or several matches is an error, never a guess.

In [ ]:
#| export
def find_evidence(
    rows: Sequence[EvidenceRow],
    *,
    series_id: str,
    model_version: str,
    origin_day: int,
    horizon: int,
    metric: str,
) -> EvidenceRow:
    """Return the single row matching every key, or raise LookupError."""
    matches = [
        row
        for row in rows
        if (row.series_id, row.model_version, row.origin_day, row.horizon, row.metric)
        == (series_id, model_version, origin_day, horizon, metric)
    ]
    if len(matches) != 1:
        raise LookupError(f"expected exactly one evidence row, found {len(matches)}")
    return matches[0]

## Example: fixture to evidence row

Load the sample series, forecast day 8, score it, and store the score as an evidence row.

In [ ]:
from pathlib import Path

from agentic_forecast_review.fixtures import (
    load_observations,
    seasonal_naive,
    split_at_origin,
)
from agentic_forecast_review.metrics import mae

history, target = split_at_origin(
    load_observations(Path("data/synthetic_weekly_01.csv")), origin_day=7
)
value = mae([target[0].value], [seasonal_naive(history, period=7, horizon=1)])
row = metric_evidence(
    series_id=target[0].series_id,
    model_version="seasonal_naive_v0",
    origin_day=7,
    horizon=1,
    metric="mae",
    value=value,
)
found = find_evidence(
    (row,),
    series_id=row.series_id,
    model_version="seasonal_naive_v0",
    origin_day=7,
    horizon=1,
    metric="mae",
)
print(row.row_id, "=", row.value)
assert value == 3.0 and found is row
assert row.row_id == "synthetic_weekly_01__seasonal_naive_v0__origin_7__h1__mae"